# Chapitre 4 — ResNet-18 : densité mammaire

**🎯 Objectif :** entraîner un réseau **connu** (ResNet-18 pré-entraîné ImageNet) en transfer learning pour classer la **densité mammaire** (BI-RADS A/B/C/D), et **comprendre** pourquoi quelques réglages (learning rate, équilibrage des classes, découpe des données) font la différence entre un entraînement qui apprend et un qui *collapse*.
**⏱ Durée :** définition du modèle instantanée ; entraînement de démo ~quelques min sur GPU **si** les données sont présentes (sinon les cellules d'entraînement s'auto-désactivent).

Premier vrai entraînement, avec un réseau **connu** : ResNet-18 pré-entraîné
ImageNet, adapté en **classifieur multiclasse** de la **densité mammaire** (BI-RADS
A / B / C / D). C'est le réflexe de base avant des architectures spécialisées
comme GMIC (ch6).

Ce notebook **entraîne sur les données RSNA** (téléchargées au ch1 dans `data/in/`,
prétraitées au ch3 dans `data/work/`). Si elles sont absentes, les cellules
d'entraînement s'auto-désactivent proprement — la définition du modèle, elle, tourne
toujours. Les valeurs qu'on propose (LR, batch…) sont des **points de départ** : tu es
invité à les changer et observer l'effet, c'est tout l'intérêt d'un notebook.

In [ ]:
import os
import numpy as np, pandas as pd
import torch, torch.nn as nn
import cv2
from course_utils import flowchart, data_in, preprocess_dir, pick_device, dataloader_kwargs

DEVICE = pick_device(verbose=False)

# ────────────────────────────────────────────────────────────────────────────
DATASET = 'rsna_sample'      # <── LE CURSEUR, le MÊME qu'au ch3 : 'rsna_sample' (démo)
#                                   ou 'rsna' (complet). Il doit désigner ce que le ch3
#                                   a réellement prétraité.
# ────────────────────────────────────────────────────────────────────────────

# train.csv : le ch1 le télécharge TOUJOURS à cet emplacement unique, quel que soit DATASET —
# c'est lui qui porte la densité de TOUTES les images. (Le train.csv de data/in/rsna_sample est
# un manifeste filtré propre au flux du ch3, PAS une autre copie de celui-ci : voir ch3.)
TRAIN_CSV = data_in('rsna', 'train.csv')
# Images : le MÊME appel que la sortie du ch3 (course_utils.preprocess_dir) -> écrivain et
# lecteur ne peuvent pas pointer vers deux dossiers différents.
IMG_DIR = preprocess_dir(DATASET, 'cropped_images')
# isdir(IMG_DIR) : preprocess_dir() construit un chemin par concaténation et renvoie TOUJOURS
# une chaîne — seul un test d'existence sur disque dit quelque chose.
DATA_OK = os.path.isfile(TRAIN_CSV) and os.path.isdir(IMG_DIR)
print('Device   :', DEVICE, '| jeu de données :', DATASET)
print('train.csv:', TRAIN_CSV if os.path.isfile(TRAIN_CSV) else f'{TRAIN_CSV} (absent -> exécute le ch1)')
print('images   :', IMG_DIR if os.path.isdir(IMG_DIR)
      else f'{IMG_DIR} (absent -> exécute le ch3 avec le MÊME DATASET)')
print('DATA_OK  :', DATA_OK, '' if DATA_OK else '-> entraînement désactivé (exécute le ch1 puis le ch3)')

# Carte du chapitre (vue d'ensemble) — chaque étape est détaillée dans les sections qui suivent.
flowchart([
    'ResNet-18 deja entraine sur ImageNet (photos couleur, 1000 classes)',
    'On adapte ses 2 extremites : entree 1 canal (gris), sortie 4 densites A/B/C/D',
    'Dataset : chaque image + son label de densite (train.csv)',
    'Reglages qui evitent le collapse : classes equilibrees, learning rate prudent',
    'Boucle : predire -> mesurer l erreur (loss) -> retropropager -> ajuster les poids',
], title='Ch4 — ResNet-18 densite')

<cell_type>markdown</cell_type>## Le modèle : partir d'un réseau déjà entraîné (transfer learning)

**L'idée.** ResNet-18 est un réseau de convolution classique. Celui qu'on charge a
déjà été entraîné sur **ImageNet** (1,2 million de photos, 1000 catégories : chiens,
voitures, bateaux…). Au fil de cet entraînement, ses premières couches ont appris à
détecter des motifs **génériques** — contours, textures, coins, dégradés — qui ne sont
propres ni aux chiens ni aux voitures : ce sont les briques de base de *toute* image.
Plutôt que de réapprendre ça de zéro sur nos quelques milliers de mammographies (ce qui
demanderait bien plus de données), on **réutilise** ces briques et on ne réajuste que ce
qui est spécifique à notre tâche. C'est le **transfer learning**.

> 🎥 Si l'architecture ResNet elle-même (blocs résiduels, la fameuse *skip connection*)
> t'intrigue, cette vidéo la décortique bien : <https://www.youtube.com/watch?v=w1UsKanMatM>.
> Ce n'est **pas** nécessaire pour la suite — ici on le traite comme une boîte déjà douée
> pour voir, qu'on adapte à ses deux extrémités.

Le réseau tel quel ne colle pas à notre problème à **deux endroits**, l'entrée et la
sortie — le corps du réseau, lui, reste intact :

**1. L'entrée — 3 canaux → 1 canal.** ImageNet, ce sont des photos **couleur** : la
première convolution (`conv1`) attend donc 3 canaux (Rouge, Vert, Bleu). Une mammographie
est en **niveaux de gris** : 1 seul canal. On doit remplacer `conv1` par une version qui
accepte 1 canal — mais en **gardant exactement les mêmes hyperparamètres** (taille de
filtre 7×7, stride 2, padding 3, 64 filtres en sortie), sinon les dimensions ne
correspondraient plus à la couche suivante et tout le reste casserait. Et au lieu de
repartir de poids aléatoires, on **moyenne les 3 filtres RGB** appris sur ImageNet en un
seul : le filtre 1-canal obtenu sait déjà repérer des contours, on ne jette pas ce savoir.

**2. La sortie — 1000 classes → 4 classes.** La dernière couche (`fc`, *fully-connected*)
d'un ResNet ImageNet produit **1000 scores** (un par catégorie ImageNet). Nous, on veut
**4 scores**, un par niveau de densité BI-RADS (A/B/C/D). On la remplace donc par une
couche linéaire neuve avec le **même nombre d'entrées** (512 pour ResNet-18, récupéré
automatiquement via `fc.in_features`) mais **4 sorties**. Contrairement à `conv1`, cette
couche démarre avec des poids **aléatoires**, et c'est **voulu** : « densité mammaire »
n'a rien à voir avec les 1000 classes d'ImageNet, il n'y a aucun savoir à réutiliser ici —
c'est précisément ce que l'entraînement va lui apprendre.</cell id="12cb2bd2">

In [2]:
from torchvision.models import resnet18, ResNet18_Weights

def build_model(n_classes=4, pretrained=True):
    """ResNet-18 ImageNet adapté à nos mammographies : entrée 1 canal, sortie n_classes.

    Poids ImageNet (transfer learning) si disponibles ; sinon **init aléatoire**
    (ex. hors-ligne : les poids se téléchargent depuis internet la 1re fois).
    Pour un simple bench de vitesse, `pretrained=False` évite tout accès réseau.
    """
    try:
        m = resnet18(weights=ResNet18_Weights.DEFAULT if pretrained else None)
    except Exception as e:                       # pas de réseau / poids non mis en cache
        print(f"⚠️ Poids ImageNet non téléchargeables ({type(e).__name__}) — init aléatoire (hors-ligne).")
        m = resnet18(weights=None)

    # --- Adaptation 1 : l'ENTRÉE, 3 canaux (RGB ImageNet) -> 1 canal (mammo niveaux de gris) ---
    # conv1.weight a la forme (64, 3, 7, 7) = 64 filtres, chacun sur 3 canaux couleur.
    # On moyenne les 3 canaux -> (64, 1, 7, 7) : un filtre 1-canal qui HÉRITE de ce que les
    # filtres RGB avaient appris (détecter des contours) au lieu de repartir de zéro.
    w = m.conv1.weight.data.mean(dim=1, keepdim=True)
    # On remplace conv1 par une couche 1-canal AVEC LES MÊMES hyperparamètres (7x7, stride 2,
    # padding 3, 64 sorties) : le reste du réseau attend une sortie de conv1 identique, donc
    # tout ce qui change c'est le nombre de canaux d'entrée — les dimensions en aval sont préservées.
    m.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
    m.conv1.weight.data = w                      # injecte les poids moyennés (sinon : aléatoires)

    # --- Adaptation 2 : la SORTIE, 1000 classes ImageNet -> n_classes (4 densités A/B/C/D) ---
    # fc est la couche linéaire finale. in_features = 512 pour ResNet-18 (récupéré dynamiquement) ;
    # on garde ces 512 entrées mais on passe à n_classes sorties. Cette couche neuve démarre
    # ALÉATOIRE, à dessein : c'est elle qui doit apprendre NOTRE tâche (rien à réutiliser d'ImageNet).
    m.fc = nn.Linear(m.fc.in_features, n_classes)
    return m

model = build_model().to(DEVICE)
print('ResNet-18 (1 canal, 4 classes) :', f'{sum(p.numel() for p in model.parameters())/1e6:.1f} M params')
# Test forward sur une image factice
with torch.no_grad():
    out = model(torch.randn(2, 1, 512, 512, device=DEVICE))
print('sortie logits :', tuple(out.shape), '(batch, 4 classes)')

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /home/deep-piste/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 44.7M/44.7M [00:07<00:00, 6.14MB/s]


ResNet-18 (1 canal, 4 classes) : 11.2 M params
sortie logits : (2, 4) (batch, 4 classes)


## Le dataset (densité)

Les PNG chargés ici sortent **déjà** du prétraitement du ch3 : cadrés, vues droites
retournées, tous en **2944×1920 uint8**. Ils sont donc déjà de taille uniforme — deux
traitements restent pourtant appliqués **au chargement**, et ce ne sont pas des redites du
ch3 :

- **Redimensionnement 2944×1920 → 512×512.** Le 2944×1920 vise **GMIC** (ch6/7), qui exploite
  la pleine résolution. Ici la densité mammaire est une tâche **grossière à 4 classes** :
  512×512 suffit largement, tout en divisant par ~20 le nombre de pixels → entraînement bien
  plus rapide et qui tient en mémoire. C'est un **sous-échantillonnage volontaire** propre à ce
  chapitre, **pas** une correction de tailles qui varieraient (elles sont déjà identiques).
- **Normalisation z-score (par image).** Le ch3 a **délibérément laissé** cette étape pour le
  chargement : centrer-réduire produit des valeurs négatives, impossibles à stocker dans un PNG
  uint8. Or un réseau converge mieux avec une entrée centrée, d'échelle ~1.

### Où faire le z-score : le curseur `Z_SCORE`

Cette normalisation est recalculée **à chaque chargement de chaque image** — c'est donc la
seule étape du pipeline qu'on a un intérêt à déplacer. Deux endroits possibles, **même
calcul** :

| `Z_SCORE` | Qui calcule | Sur quoi |
|---|---|---|
| `'cpu'` | les workers du `DataLoader` | une image à la fois, avant l'empilement |
| `'gpu'` | la boucle d'entraînement | le batch entier, après le transfert |

Le **chapitre 5** *mesure* lequel est le plus rapide sur ta machine ; ce chapitre-ci
l'**applique**. Par défaut `'gpu'` : les workers ne font plus que décoder. Bascule sur
`'cpu'` si le bench dit que c'est mieux chez toi — le résultat de l'entraînement, lui, est
inchangé.

> ⚠️ Le piège : la normalisation doit être appliquée **à l'entraînement ET à l'évaluation**.
> D'où l'unique helper `to_device()` plus bas, par lequel passent les deux boucles.

On mappe ensuite la densité `A/B/C/D → 0/1/2/3`, et on ne garde que les lignes dont l'image
existe (cf. `img_path`) et la densité est renseignée.

In [ ]:
from torch.utils.data import Dataset

DENS = {'A': 0, 'B': 1, 'C': 2, 'D': 3}

# ────────────────────────────────────────────────────────────────────────────
Z_SCORE = 'gpu'    # <── OÙ se fait la normalisation : 'cpu' (dans les workers, une image à
#                        la fois) ou 'gpu' (sur le batch entier, après transfert).
#                        Le CALCUL est identique — seul l'endroit change. Le ch5 MESURE
#                        lequel est le plus rapide sur ta machine ; ici on l'APPLIQUE.
# ────────────────────────────────────────────────────────────────────────────

def gpu_standardize(x):
    """z-score par image, appliqué au batch entier là où vit le tenseur (GPU).

    dim=(1,2,3) : moyenne sur canal+hauteur+largeur mais PAS sur la dimension 0 (le batch),
    donc chaque image garde ses propres statistiques — exactement comme la version CPU
    ci-dessous, qui traite une image à la fois. unbiased=False = convention numpy (÷N).
    """
    x = x.float()
    m = x.mean(dim=(1, 2, 3), keepdim=True)
    s = x.std(dim=(1, 2, 3), keepdim=True, unbiased=False).clamp(min=1e-5)
    return (x - m) / s

def img_path(pid, iid):
    # IMG_DIR est TOUJOURS une sortie du prétraitement (ch3), organisée par patient : une
    # seule convention de nommage, <IMG_DIR>/<pid>/<iid>.png (set complet ou échantillon = même
    # format). On vérifie quand même l'existence sur disque, car le CSV liste toutes les images
    # de la compétition alors que seul un sous-ensemble a pu être prétraité (ex. la mini-démo).
    p = os.path.join(IMG_DIR, str(pid), f'{iid}.png')
    return p if os.path.isfile(p) else None

class DensityDataset(Dataset):
    def __init__(self, rows, size=512, cpu_norm=(Z_SCORE == 'cpu')):
        self.rows, self.size, self.cpu_norm = rows, size, cpu_norm
    def __len__(self): return len(self.rows)
    def __getitem__(self, i):
        path, label = self.rows[i][0], self.rows[i][1]   # rows = (path, label, patient_id)
        img = cv2.imread(path, cv2.IMREAD_UNCHANGED).astype(np.float32)
        # Les PNG du ch3 font 2944x1920 (taille GMIC) : on RÉDUIT à 512x512 — assez pour une
        # tâche grossière à 4 classes, et ~20x moins de pixels donc entraînement bien plus léger.
        img = cv2.resize(img, (self.size, self.size))
        # z-score par image : normalisation que le ch3 a laissée au chargement (elle donne des
        # floats négatifs, non stockables en PNG uint8). Entrée centrée -> le réseau converge mieux.
        # En mode 'gpu' on ne la fait PAS ici : le worker ne décode que, et la boucle
        # d'entraînement appelle gpu_standardize() sur le batch complet.
        if self.cpu_norm:
            img = (img - img.mean()) / max(img.std(), 1e-5)
        # img[None] insère un axe de taille 1 EN TÊTE : (H, W) -> (1, H, W). Le None dans une
        # indexation numpy ne sélectionne aucun élément, il CRÉE une dimension (alias de np.newaxis) ;
        # c'est une simple nouvelle vue, sans copie des pixels. Cet axe est le CANAL : le modèle veut
        # du (C, H, W) et sa conv1 attend justement 1 canal (cf. section « Le modèle »).
        return torch.tensor(img[None], dtype=torch.float32), label

samples, labels = [], []
if DATA_OK:
    dfc = pd.read_csv(TRAIN_CSV)
    dfc = dfc[dfc['density'].isin(DENS)]
    for pid, iid, dens in dfc[['patient_id', 'image_id', 'density']].itertuples(index=False):
        p = img_path(pid, iid)
        if p:
            samples.append((p, DENS[dens], pid)); labels.append(DENS[dens])   # on garde patient_id
        if len(samples) >= 4000:    # sous-ensemble pour une démo rapide
            break
    print('échantillons :', len(samples), '| répartition classes :', np.bincount(labels, minlength=4).tolist())
    print(f"z-score : {Z_SCORE.upper()}",
          '(dans les workers)' if Z_SCORE == 'cpu' else '(sur le batch, après transfert)')
else:
    print('Données absentes -> dataset vide (cellule d entraînement sautée).')

> 📎 **Aparté déplacé.** Une question de *performance* se pose ici : sur machine puissante,
> qui doit faire le z-score — les workers CPU, ou le GPU sur le batch entier ? Elle n'a rien à
> voir avec la densité mammaire, et la mesurer coûte plusieurs minutes. Elle a donc son propre
> notebook : **[`05_bench_normalisation.ipynb`](05_bench_normalisation.ipynb)**, qui mesure
> la pipeline complète (décodage réel → z-score → forward/backward) en CPU-norm vs GPU-norm.
>
> À lire après ce chapitre — le principe y sert directement au **ch7** (GMIC en 2944×1920, où
> déplacer la normalisation sur GPU a fait passer l'utilisation GPU de **49 % à 91 %**).

## Pourquoi ces réglages ? (les pièges du collapse)

Les valeurs de la boucle ci-dessous ne sont pas magiques : chacune répond à un piège
précis, et **chacune correspond à une ligne de la cellule suivante**. Le meilleur moyen
de les comprendre, c'est de les **casser** et de regarder ce qui arrive à la loss et aux
prédictions :

- **LR 1e-5** (`Adam(..., lr=1e-5)`) : un learning rate trop haut fait *collapser* le
  réseau — toutes les prédictions deviennent constantes (la classe majoritaire). Essaie
  `1e-3` pour le voir arriver.
- **`WeightedRandomSampler`** rééquilibre des classes inégales (ici 408/1601/1811/180).
  Si tu l'utilises, garde une **`CrossEntropyLoss` simple** : cumuler sampler *et* loss
  pondérée compense deux fois et déséquilibre dans l'autre sens.
- **Split au niveau patient** (`assert train_pids.isdisjoint(...)`), jamais au niveau
  image : deux vues d'un même patient réparties entre train et test **gonflent** le score
  (fuite de données). C'est le piège le plus sournois car le modèle a l'air excellent.

> 🩹 **Piège voisin, non illustré ici** (aucune ligne du notebook ne l'utilise, mais il
> coûte une soirée quand on tombe dessus) : ne pas activer `channels_last` avec une entrée
> **1 canal** — ça déclenche une erreur CUDA.

In [ ]:
from torch.utils.data import DataLoader, WeightedRandomSampler

def to_device(x, y=None):
    """Transfère sur DEVICE et, en mode 'gpu', applique le z-score que les workers n'ont pas fait.

    Un SEUL point de passage pour train ET eval : c'est volontaire. Normaliser à
    l'entraînement mais pas à l'évaluation (ou l'inverse) donnerait un modèle qui voit deux
    distributions différentes — l'accuracy s'écroulerait sans qu'aucune erreur ne soit levée.
    Le piège classique quand on déplace une normalisation.
    """
    x = x.to(DEVICE)
    if Z_SCORE == 'gpu':
        x = gpu_standardize(x)
    return x if y is None else (x, y.to(DEVICE))

if DATA_OK and samples:
    # Split AU NIVEAU PATIENT (pas image) : toutes les vues d'un même patient_id
    # tombent du même côté -> évite la fuite de données (data leakage).
    rng = np.random.default_rng(42)
    pids = np.array(sorted({r[2] for r in samples}))
    rng.shuffle(pids)
    cut = int(0.8 * len(pids))
    train_pids = set(pids[:cut].tolist())
    tr_rows = [r for r in samples if r[2] in train_pids]
    te_rows = [r for r in samples if r[2] not in train_pids]
    tr_labels = np.array([r[1] for r in tr_rows])
    assert train_pids.isdisjoint({r[2] for r in te_rows}), 'fuite patient !'

    # sampler équilibré : poids inversement proportionnel à la fréquence de classe
    freq = np.bincount(tr_labels, minlength=4)
    w = (1.0 / np.maximum(freq, 1))[tr_labels]
    sampler = WeightedRandomSampler(torch.tensor(w, dtype=torch.double), len(w), replacement=True)

    # dataloader_kwargs() règle num_workers / pin_memory selon la machine (voir course_utils).
    tr = DataLoader(DensityDataset(tr_rows), **dataloader_kwargs(batch_size=16, sampler=sampler))
    te = DataLoader(DensityDataset(te_rows), **dataloader_kwargs(batch_size=16, shuffle=False))
    opt = torch.optim.Adam(model.parameters(), lr=1e-5)
    loss_fn = nn.CrossEntropyLoss()

    for epoch in range(2):                      # démo courte ; monter pour un vrai run
        model.train()
        for x, y in tr:
            x, y = to_device(x, y)              # transfert + z-score GPU si Z_SCORE == 'gpu'
            logits = model(x)
            loss = loss_fn(logits, y)
            opt.zero_grad(); loss.backward(); opt.step()
        model.eval(); correct = total = 0
        with torch.no_grad():
            for x, y in te:
                pred = model(to_device(x)).argmax(1).cpu()   # MÊME normalisation qu'au train
                correct += (pred == y).sum().item(); total += len(y)
        print(f'epoch {epoch}  loss={loss.item():.3f}  test_acc={correct/total:.3f}'
              f'  (z-score {Z_SCORE})')
else:
    print('Entraînement sauté (DATA_OK =', DATA_OK, ').')